# 02 · 코호트 정의 — 데이터 구조와 품질 필터

## 데이터 구조

Ext-PPG는 **세그먼트 단위**로 배포된다.

| 층 | 단위 | 개수 | 담긴 것 |
|---|---|---|---|
| 환자 | `subject_id` | 6,189 | age · gender · height · weight · ethnicity · `icd9` · `icd10_truncated` · `strat_fold`(0–9) |
| 레코드 | `record_id` | 92,077 | 한 번의 입원 중 연속 기록. 원본 MIMIC-III Waveform과 같은 번호 체계 |
| 세그먼트 | `.hea` + `.dat` | 6,399,754 | 30초 · 125 Hz PLETH · `event_rhythm` · SQI 4종 · SBP/DBP · HR · RR |

파형 파일 하나에 `metadata.csv`의 한 행이 대응하고, 그 행에 품질 지표·리듬·인구통계·진단코드가
함께 들어 있다. 파형과 라벨이 같은 자리에 있다는 것이 이 데이터셋을 1단계에 쓰는 이유다.

## 품질 지표(SQI)는 데이터셋이 제공하는 값이다

우리가 계산하지 않았다. 데이터셋이 30초를 **10초씩 세 구간**(0–10 · 10–20 · 20–30초)으로 나눠
각 구간에 SQI를 매기고, 3원소 벡터 `vector_10s_pleth_sqi`로 배포한다.
판정 알고리즘은 **Orphanidou 등(2015)**이며 세 가지를 본다 —
심박수가 생리적 범위 안인가, 박동 간격이 일관되는가, 평균 박동 **템플릿과의 상관**으로 형태가 일관되는가.

```
vector_10s_pleth_sqi = [q1, q2, q3],  qj ∈ {+1, 0}
  +1  세 검사를 모두 통과
   0  하나 이상 실패 (사용은 가능)
  음수 계산 자체가 불가 (−14 RR 간격 없음, −17 피크 부족) → 배포 전 이미 제외됨
```

우리 필터는 **세 구간이 모두 +1**인 세그먼트만 통과시킨다.

> Orphanidou 지표가 보는 것은 박동 검출 가능성과 박동 간 일관성이다.
> c·d파나 반사파 봉우리 같은 **미세 형태의 신뢰도는 이 필터가 보증하지 않는다.**
> 그것은 별도의 검출률(`cd_rate` · `ri_rate` · `lvet_rate`)로 다룬다.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from ppg_fm import paths, features as F

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("프로젝트", paths.ROOT)
print("데이터  ", paths.data_root())
from ppg_fm.report import Report
rep = Report("01_dataset/02_cohort_definition")
print("산출물 →", rep.dir)

## 1. SQI 분포 — 필터가 무엇을 남기고 무엇을 버리는가

In [ ]:
from ppg_fm.data import cohort

dist = cohort.sqi_distribution()
rep.table(dist, "sqi_distribution.csv", "SQI 벡터별 세그먼트 수")
dist

In [ ]:
hq = dist[dist.hq].n_segment.sum()
print(f"고품질 [1,1,1] : {hq:,} ({100*hq/dist.n_segment.sum():.1f}%)")
print(f"제외          : {dist[~dist.hq].n_segment.sum():,}")

## 2. 세그먼트 인덱스 생성

`metadata.csv`를 한 번 훑어 세그먼트 인덱스를 만든다. `hq` 플래그를 여기서 붙인다.

In [ ]:
idx_path = paths.interim("seg_index.parquet")
if not idx_path.exists():
    cohort.build_index()
idx = pd.read_parquet(idx_path, columns=["subject", "rhythm", "hq", "has_abp"])
print(f"세그먼트 {len(idx):,} · 환자 {idx.subject.nunique():,}")
idx.head()

## 3. 추출 계획 — 환자당 세그먼트 10개

환자마다 기록 전체에서 **균등 간격**으로 10개를 고른다. 앞쪽만 쓰면 입원 초기에 치우친다.

**리듬은 제한하지 않는다.** 정상동율동(SR)만 남기던 조건이 고품질 환자 6,114명 중
867명(AF 452 · VPACE 166 · STACH 112 …)을 통째로 배제하고 있었다.
심장 질환을 보는 분석에서 부정맥 환자를 빼고 있었던 셈이다.

In [ ]:
excluded = idx[idx.hq].groupby("subject").rhythm.apply(lambda s: (s == "SR").sum() == 0)
print(f"SR 세그먼트를 하나도 갖지 않는 환자: {int(excluded.sum()):,}명")
lost = idx[idx.hq & idx.subject.isin(excluded[excluded].index)]
lost.groupby("subject").rhythm.agg(lambda s: s.mode().iat[0]).value_counts().head(8)

In [ ]:
plan_path = paths.interim("extract_plan.csv")
if not plan_path.exists():
    cohort.build_plan(n_per_patient=10)
plan = pd.read_csv(plan_path, dtype={"subject": str})
print(f"계획 세그먼트 {len(plan):,} · 환자 {plan.subject.nunique():,}")

## 4. 코호트가 확정되는 경로

In [ ]:
flow = cohort.cohort_flow()
rep.table(flow, "cohort_flow.csv", "코호트 확정 경로")
flow

### 최종 1단계 코호트

- 환자 **6,113명**
- 세그먼트 59,113개 · 박동 2,361,435개
- 환자당 박동 중앙값 395개 (최소 22)
- 리듬 제한 없음

In [ ]:
feat = paths.interim("patient_features_v2.csv")
if feat.exists():
    P = pd.read_csv(feat, dtype={"subject": str})
    print(P[["n_beats", "HR", "age", "sr_frac", "af_frac", "abp_frac"]].describe().round(2).to_string())

## 산출물

In [ ]:
rep.done("데이터 구조 · SQI 필터 · 코호트 확정 경로")
rep.summary()